# Evaluate the Decision Tree

- You can assess the quality of your model by evaluating how well it performs on the testing data. Because the model was not trained on these data, this represents an objective assessment of the model.

A _confusion matrix_ gives a useful breakdown of predictions versus known values. It has four cells which represent the counts of:

- _True Negatives_ (TN) — model predicts negative outcome & known outcome is negative
- _True Positives_ (TP) — model predicts positive outcome & known outcome is positive
- _False Negatives_ (FN) — model predicts negative outcome but known outcome is positive
- _False Positives_ (FP) — model predicts positive outcome but known outcome is negative.
  
These counts (`TN`, `TP`, `FN` and `FP`) should sum to the number of records in the testing data, which is only a subset of the flights data. You can compare to the number of records in the tests data, which is `flights_test.count()`.

Note: These predictions are made on the testing data, so the counts are smaller than they would have been for predictions on the training data.

## Instructions

- Create a confusion matrix by counting the combinations of `label` and `prediction`. Display the result.
- Count the number of True Negatives, True Positives, False Negatives and False Positives.
- Calculate the accuracy.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [ ]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [ ]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [2]:
# Read data from CSV file
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/2_DecisionTree/dataset/flights.csv',
                         sep=',',
                         header=True,
                         inferSchema=True,
                         nullValue='NA')

In [3]:
flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights['mile'] * 1.60934, 0)).drop(flights['mile'])\
.withColumn('label', (flights.delay > 15).cast('integer'))
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol='carrier', outputCol='carrier_idx')
indexer_model = indexer.fit(flights)
flights = indexer_model.transform(flights)
flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=[
    'mon', 'dom', 'dow', 'carrier_idx', 'org_idx', 'km', 'depart', 'duration'
	], outputCol='features')
flights = assembler.transform(flights)

# Split into training and testing sets in a 80:20 ratio
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed = 43)

# Import the Decision Tree Classifier class
from pyspark.ml.classification import DecisionTreeClassifier

# Create a classifier object and fit to the training data
tree = DecisionTreeClassifier()
tree_model = tree.fit(flights_train)

# Create predictions for the testing data and take a look at the predictions
prediction = tree_model.transform(flights_test)
prediction.select('label', 'prediction', 'probability').show(5, False)

In [ ]:
# Create a confusion matrix
prediction.groupBy(____, 'prediction').____().____()

# Calculate the elements of the confusion matrix
TN = prediction.filter('prediction = 0 AND label = prediction').count()
TP = prediction.____('____ AND ____').____()
FN = prediction.____('____ AND ____').____()
FP = prediction.____('____ AND ____').____()

# Accuracy measures the proportion of correct predictions
accuracy = ____
print(accuracy)